# Project 2 Colab Runner

Run this notebook after cloning the repository in Colab. It mounts Google Drive, installs dependencies, logs into Weights & Biases, and runs training in resumable chunks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If you opened this notebook from GitHub before cloning, set REPO_URL and run this cell.
# If the repo is already cloned, this cell just changes into it.
import os
from pathlib import Path

REPO_URL = "https://github.com/jyun-chae/skku-2-openai_pa2.git"
REPO_DIR = Path('/content/project02')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd /content/project02

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
# Login to W&B without saving the API key in the notebook.
import getpass
import os
import wandb

WANDB_API_KEY = getpass.getpass('W&B API key: ')
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY)

## Configure One Training Chunk

Set `STAGE` to `256`, `512`, or `1024`. Put the corresponding training zip in Drive, for example `/content/drive/MyDrive/project02/data/train_50k_256.zip`.

In [ ]:
from pathlib import Path

STAGE = "256"  # "256", "512", or "1024"
MAX_STEPS = 50000
SAVE_EVERY_STEPS = 1000
A100_BATCH_SIZE = {"256": 12, "512": 6, "1024": 2}
A100_GRAD_ACCUM_STEPS = {"256": 6, "512": 4, "1024": 4}
A100_NUM_WORKERS = {"256": 8, "512": 8, "1024": 6}
BATCH_SIZE = A100_BATCH_SIZE[STAGE]
GRAD_ACCUM_STEPS = A100_GRAD_ACCUM_STEPS[STAGE]
NUM_WORKERS = A100_NUM_WORKERS[STAGE]
PRECISION = "bf16"
PATH_REG_WEIGHT = 2.0
PATH_REG_SHRINK = 8
PATH_REG_EVERY = 4
FID_EVERY_STEPS = 1000
WANDB_MODE = "online"  # "online", "offline", or "disabled"

DRIVE_ROOT = Path('/content/drive/MyDrive/project02')
DATA_ZIP = DRIVE_ROOT / 'data' / f'train_50k_{STAGE}.zip'
VALID_ZIP = DRIVE_ROOT / 'data' / f'valid_10k_{STAGE}.zip'
DATA_DIR = Path('/content/project02_data') / f'train_50k_{STAGE}'
VALID_DIR = Path('/content/project02_data') / f'valid_10k_{STAGE}'
RUN_DIR = DRIVE_ROOT / 'runs' / f'stylegan_{STAGE}'
CONFIG = Path('configs') / f'stylegan_{STAGE}.yaml'

RUN_DIR.mkdir(parents=True, exist_ok=True)
print('config:', CONFIG)
print('data_zip:', DATA_ZIP)
print('data_dir:', DATA_DIR)
print('valid_zip:', VALID_ZIP)
print('valid_dir:', VALID_DIR)
print('run_dir:', RUN_DIR)
assert DATA_ZIP.exists(), f'Missing training zip: {DATA_ZIP}'
assert VALID_ZIP.exists(), f'Missing validation zip for FID: {VALID_ZIP}'

In [ ]:
# Unzip once to Colab local disk. Training from local files is usually faster than reading from Drive zip.
DATA_DIR.mkdir(parents=True, exist_ok=True)
has_images = any(p.suffix.lower() in ('.png', '.jpg', '.jpeg') for p in DATA_DIR.rglob('*'))
if not has_images:
    !unzip -q {DATA_ZIP} -d {DATA_DIR}
else:
    print('already unzipped:', DATA_DIR)

VALID_DIR.mkdir(parents=True, exist_ok=True)
has_valid_images = any(p.suffix.lower() in ('.png', '.jpg', '.jpeg') for p in VALID_DIR.rglob('*'))
if not has_valid_images:
    !unzip -q {VALID_ZIP} -d {VALID_DIR}
else:
    print('already unzipped:', VALID_DIR)

TRAIN_DATA = DATA_DIR
FID_REAL_DATA = VALID_DIR
print('training data:', TRAIN_DATA)
print('fid real data:', FID_REAL_DATA)

## Start or Resume

For a new stage, run the first cell below. If Colab disconnects or you want another chunk, run the resume cell.

In [ ]:
# New run for the selected stage.
# For 512/1024, this automatically initializes from the previous stage's final.pt if it exists.
import subprocess
def run_and_show(cmd):
    import subprocess
    print(' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print('--- train.py stdout ---')
        print(result.stdout[-12000:])
    if result.stderr:
        print('--- train.py stderr ---')
        print(result.stderr[-12000:])
    print('exit code:', result.returncode)
    result.check_returncode()
from pathlib import Path

prev_stage = {"256": None, "512": "256", "1024": "512"}[STAGE]
init_args = []
if prev_stage is not None:
    prev_final = DRIVE_ROOT / 'runs' / f'stylegan_{prev_stage}' / 'final.pt'
    if prev_final.exists():
        init_args = ['--init-from', str(prev_final)]
        print('initializing from', prev_final)
    else:
        print('previous final checkpoint not found:', prev_final)

cmd = [
    'python', 'train.py',
    '--config', str(CONFIG),
    '--train-zip', str(TRAIN_DATA),
    '--fid-real-zip', str(FID_REAL_DATA),
    '--run-dir', str(RUN_DIR),
    '--wandb-name', f'stylegan_{STAGE}_colab',
    '--wandb-mode', WANDB_MODE,
    '--max-steps', str(MAX_STEPS),
    '--save-every-steps', str(SAVE_EVERY_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum-steps', str(GRAD_ACCUM_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--precision', PRECISION,
    '--path-regularize-weight', str(PATH_REG_WEIGHT),
    '--path-length-batch-shrink', str(PATH_REG_SHRINK),
    '--path-length-lazy-every', str(PATH_REG_EVERY),
    '--fid-every-steps', str(FID_EVERY_STEPS),
] + init_args
run_and_show(cmd)


In [ ]:
# Resume the selected stage from Google Drive latest.pt.
import subprocess
def run_and_show(cmd):
    import subprocess
    print(' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print('--- train.py stdout ---')
        print(result.stdout[-12000:])
    if result.stderr:
        print('--- train.py stderr ---')
        print(result.stderr[-12000:])
    print('exit code:', result.returncode)
    result.check_returncode()
LATEST = RUN_DIR / 'latest.pt'
assert LATEST.exists(), f'Missing checkpoint: {LATEST}'

cmd = [
    'python', 'train.py',
    '--config', str(CONFIG),
    '--train-zip', str(TRAIN_DATA),
    '--fid-real-zip', str(FID_REAL_DATA),
    '--run-dir', str(RUN_DIR),
    '--resume', str(LATEST),
    '--wandb-name', f'stylegan_{STAGE}_colab',
    '--wandb-mode', WANDB_MODE,
    '--max-steps', str(MAX_STEPS),
    '--save-every-steps', str(SAVE_EVERY_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum-steps', str(GRAD_ACCUM_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--precision', PRECISION,
    '--path-regularize-weight', str(PATH_REG_WEIGHT),
    '--path-length-batch-shrink', str(PATH_REG_SHRINK),
    '--path-length-lazy-every', str(PATH_REG_EVERY),
    '--fid-every-steps', str(FID_EVERY_STEPS),
]
run_and_show(cmd)


## Generate Samples From Latest Checkpoint

In [ ]:
SAMPLE_OUT = RUN_DIR / f'sample_{STAGE}.png'
!python generate.py --ckpt {RUN_DIR / 'latest.pt'} --out {SAMPLE_OUT} --n 16 --nrow 4
print('saved:', SAMPLE_OUT)

## Export Submission ONNX

Run this after the 1024 stage is trained enough. If `onnx` is missing in the runtime, rerun the install cell.

In [ ]:
ONNX_OUT = RUN_DIR / 'submission.onnx'
!python export_onnx.py --ckpt {RUN_DIR / 'latest.pt'} --out {ONNX_OUT}
print('saved:', ONNX_OUT)